# ⚽ Division FC - World Cup 2022 Final Analysis

Professional soccer visualizations with "Midnight Aurora" branding.

**Data Source:** StatsBomb Open Data (CC BY 4.0)

## 1. Setup

In [ ]:
# Add project root to path
import sys
sys.path.insert(0, '..')

# Import our modules
from src.data.loader import (
    StatsBombLoader, 
    get_top_players_by_stat, 
    get_goals, 
    get_shots, 
    calculate_xg
)
from src.export.instagram import InstagramExporter

print("✅ Setup complete!")

## 2. Load World Cup Final Data

All IDs are dynamically sourced from the API - nothing is hardcoded.

In [ ]:
# Initialize loader
loader = StatsBombLoader()

# Load World Cup Final (dynamic)
events, match_info = loader.load_match_by_criteria(
    competition_name="World Cup",
    stage="Final"
)

# Display match info (all from data)
print(f"🏆 {match_info['home_team']} vs {match_info['away_team']}")
print(f"📊 Score: {int(match_info['home_score'])} - {int(match_info['away_score'])}")
print(f"📅 Date: {match_info['match_date']}")
print(f"✅ Loaded {len(events)} events")

## 3. Explore the Data

In [ ]:
# View goals (extracted from data)
goals = get_goals(events)

print(f"⚽ Goals ({len(goals)} total):")
for _, goal in goals.iterrows():
    player = goal['player_name'].split()[-1]
    minute = int(goal['minute'])
    team = goal['team_name']
    print(f"   {minute}' - {player} ({team})")

In [ ]:
# View top players by stat (from data)
home_team = match_info['home_team']
away_team = match_info['away_team']

print(f"🎯 Top Shooters:")
for team in [home_team, away_team]:
    shooters = get_top_players_by_stat(events, team, stat='shots', n=2)
    print(f"   {team}: {[p.split()[-1] for p in shooters]}")

## 4. Generate All Visualizations

Run the full analysis script to generate all visualizations.

In [ ]:
%run ../examples/world_cup_analysis.py

## 5. View Generated Images

Display the generated visualizations.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

images_dir = Path('../output/images')

# List all generated images
print("📸 Generated images:")
for img in sorted(images_dir.glob('*.png')):
    print(f"   - {img.name}")

In [ ]:
# Display match summary
display(Image(filename='../output/images/match_summary.png', width=500))

In [ ]:
# Display zone control
display(Image(filename='../output/images/zone_control.png', width=600))

## 6. Export for Instagram

In [ ]:
%run ../examples/instagram_export.py

In [ ]:
# Display Instagram-ready post
display(Image(filename='../output/images/instagram_post.png', width=400))

## 7. Custom Visualization

Create your own visualization with the data.

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import VerticalPitch

# Brand colors
COLORS = {
    'background': '#0D1117',
    'cyan': '#00FFCC',
    'magenta': '#FF66B2',
    'gold': '#FFD700',
    'text': '#F0F6FC',
}

# Get a player (from data)
player_name = get_top_players_by_stat(events, home_team, stat='shots', n=1)[0]
player_shots = events[
    (events['type_name'] == 'Shot') &
    (events['player_name'] == player_name)
]

# Create shot map
fig, ax = plt.subplots(figsize=(8, 10), facecolor=COLORS['background'])
ax.set_facecolor(COLORS['background'])

pitch = VerticalPitch(
    pitch_type='statsbomb',
    half=True,
    pitch_color=COLORS['background'],
    line_color='#8B949E',
    linewidth=1
)
pitch.draw(ax=ax)

# Plot shots
for _, shot in player_shots.iterrows():
    is_goal = shot['outcome_name'] == 'Goal'
    ax.scatter(
        shot['y'], shot['x'],
        s=300 if is_goal else 150,
        c=COLORS['gold'] if is_goal else COLORS['cyan'],
        marker='*' if is_goal else 'o',
        edgecolors='white',
        linewidths=1.5,
        zorder=10
    )

# Title
short_name = player_name.split()[-1]
ax.text(40, 123, f"{short_name.upper()} SHOTS", fontsize=20, fontweight='bold',
        color=COLORS['text'], ha='center')

plt.tight_layout()
plt.show()

## 📁 Output Files

All generated files are saved in:
- `output/images/` - PNG visualizations
- `output/gifs/` - Animated GIFs

---

## 🙏 Credits

- **Data:** StatsBomb Open Data (CC BY 4.0)
- **Library:** mplsoccer by Andrew Rowlinson
- **Inspiration:** Son of a Corner

---

*Division FC - Soccer Analytics* ⚽